In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
import sys
from torch.utils.data import Dataset, DataLoader, RandomSampler
import math
from collections import OrderedDict
from torch.optim import AdamW

In [2]:
#!pip install torchinfo
from torchinfo import summary

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
# Set the environment variable TOKENIZERS_PARALLELISM to 'false'
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [5]:
#!pip install transformers==4.49.0
#!pip install datasets

In [6]:
from transformers import AutoTokenizer

In [7]:
torch.__version__

'2.2.2'

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len, device):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=device)
        position = torch.arange(0., max_len,
                                device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0., d_model, 2, device=device) * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos)
        pe[:, 1::2] = torch.cos(pe_pos)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        out = self.pe[:, :x.size(1)].requires_grad_(False)
        return out

In [9]:
#del pe
pe_e = PositionalEncoding(d_model=768, dropout=0.1, max_len=512, device=device)
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [10]:
pe_e_e = pe_e(inp_tok)
pe_e_e, pe_e_e.shape

(tensor([[[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  ...,  1.0000e+00,
            0.0000e+00,  1.0000e+00],
          [ 8.4147e-01,  5.4030e-01,  8.2843e-01,  ...,  1.0000e+00,
            1.0243e-04,  1.0000e+00],
          [ 9.0930e-01, -4.1615e-01,  9.2799e-01,  ...,  1.0000e+00,
            2.0486e-04,  1.0000e+00],
          ...,
          [-7.5680e-01, -6.5364e-01, -6.9153e-01,  ...,  1.0000e+00,
            4.0971e-04,  1.0000e+00],
          [-9.5892e-01,  2.8366e-01, -9.8573e-01,  ...,  1.0000e+00,
            5.1214e-04,  1.0000e+00],
          [-2.7942e-01,  9.6017e-01, -4.1267e-01,  ...,  1.0000e+00,
            6.1457e-04,  1.0000e+00]]], device='cuda:0'),
 torch.Size([1, 7, 768]))

In [11]:
class Embed(nn.Module):
    def __init__(self, vocab_size, embed_dim, ctx_len, do, device):
        super().__init__()
        self.tok_layer = nn.Embedding(vocab_size, embed_dim)
        self.pos_layer = PositionalEncoding(embed_dim, do, ctx_len, device)
        self.norm_do = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(do)
            )
        self.device = device
        self.ctx_len = ctx_len

    def forward(self, inp):
        inp = inp.to(self.device)
        len_inp = inp.shape[-1]
        try:
            assert len_inp <= self.ctx_len
        except:
            print("Err..Errr. Error...Bro..Length of supplied text exceeds context length defined.Exiting the program now")
            sys.exit(1)
        tok_embed = self.tok_layer(inp)
        pos_embed = self.pos_layer(inp)
        embed_tok_pos = tok_embed + pos_embed
        embed_out = self.norm_do(embed_tok_pos)
        return embed_out

In [12]:
inp_tok = torch.tensor([5,8,78, 86, 78, 90, 45]).unsqueeze(0)
inp_tok, inp_tok.shape

(tensor([[ 5,  8, 78, 86, 78, 90, 45]]), torch.Size([1, 7]))

In [13]:
emb = Embed(100, 768, 512, 0.1, device).to(device)

In [14]:
emb_out = emb(inp_tok.to(device))
emb_out.shape, emb_out

(torch.Size([1, 7, 768]),
 tensor([[[-2.4344, -0.2160, -1.1357,  ...,  0.8146, -1.1258,  0.4370],
          [ 0.4459, -0.0000,  0.6194,  ...,  2.5831,  0.0535, -0.3247],
          [ 1.7692, -2.5958,  1.2576,  ...,  1.5342,  0.4727, -0.7132],
          ...,
          [ 0.1576, -2.7135, -0.2974,  ...,  1.5616,  0.5229, -0.6380],
          [-1.8427, -2.3461, -0.9518,  ...,  0.5105, -0.1308,  0.0000],
          [-0.0000,  1.9962, -1.1940,  ...,  0.2689,  0.2973,  0.8023]]],
        device='cuda:0', grad_fn=<NativeDropoutBackward0>))

In [15]:
summary(emb)

Layer (type:depth-idx)                   Param #
Embed                                    --
├─Embedding: 1-1                         76,800
├─PositionalEncoding: 1-2                --
│    └─Dropout: 2-1                      --
├─Sequential: 1-3                        --
│    └─LayerNorm: 2-2                    1,536
│    └─Dropout: 2-3                      --
Total params: 78,336
Trainable params: 78,336
Non-trainable params: 0

In [16]:
def att_mask(attention_mask, lookahead, cross_att, x=None):
    if cross_att:
        batch_dim = x[0]
        repeat = x[1]
    else:
        batch_dim = attention_mask.shape[0]
        repeat = len(attention_mask[0])

    mask =[]
    for i in range(batch_dim):
        am_interim = [attention_mask[i].tolist()] * repeat
        am_interim = torch.tensor(am_interim).unsqueeze(0)
        mask.append(am_interim)
    mask = torch.vstack(mask)
    if lookahead:
        inp_save = mask
        mask = torch.tril(torch.ones(mask.shape))
    mask = torch.where(mask == 0, -torch.inf, 0.0)
    return mask

In [17]:
attention_mask = torch.tensor([[1,1,1,1,0,0], [1,1,1,0,0,0]])
attention_mask.shape

torch.Size([2, 6])

In [18]:
mask  = att_mask(attention_mask, lookahead=False, cross_att=False, x=[2,9,6])

In [19]:
mask, mask.shape

(tensor([[[0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf],
          [0., 0., 0., 0., -inf, -inf]],
 
         [[0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf],
          [0., 0., 0., -inf, -inf, -inf]]]),
 torch.Size([2, 6, 6]))

In [20]:
class Attention(nn.Module):
    def __init__(self, embed_dim, k_dim, do, device):
        super().__init__()
        self.embed_dim = embed_dim
        self.k_dim = k_dim
        self.query = nn.Linear(embed_dim, k_dim)
        self.key = nn.Linear(embed_dim, k_dim)
        self.value = nn.Linear(embed_dim, k_dim)
        self.att_do = nn.Dropout(do)
        self.device = device

    def forward(self, qry, ky, vlu, mask):
        q = self.query(qry)
        k = self.key(ky)
        v = self.value(vlu)
        qk = (q@k.transpose(1, 2))/(self.k_dim**0.5)
        mask = mask.to(self.device)
        qk_m = qk + mask
        qk_m_smax = torch.softmax(qk_m, dim=-1)
        qk_m_smax_do = self.att_do(qk_m_smax)
        qkv = qk_m_smax_do@v
        return qkv

In [21]:
class Attention_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.num_heads = num_heads
        self.heads_list = [Attention(embed_dim, k_dim, do, device) for i in range(num_heads)]
        self.heads = nn.ModuleList(self.heads_list)
        self.lin_do = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Dropout(do)
        )

    def forward(self, q, k, v, mask):
        heads_list_out = [head(q, k, v, mask) for head in self.heads]
        att_head = torch.cat(heads_list_out, dim=-1)
        att_head_out = self.lin_do(att_head)
        return att_head_out

In [22]:
class Encoder_Block(nn.Module):
    def __init__(self, num_heads, embed_dim, k_dim, do, device):
        super().__init__()
        self.MHA = Attention_Block(num_heads, embed_dim, k_dim, do, device)
        self.mha_blk_end_layernorm = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim),
            nn.Dropout(do)
            )
        self.enc_blk_end_layernorm = nn.LayerNorm(embed_dim)

    def forward(self, args_list):  #q, k, v, mask):
        x = args_list[0]
        mask = args_list[1]

        inp_start_att_block = x
        x = self.MHA(x, x, x, mask)

        x = x + inp_start_att_block
        x = self.mha_blk_end_layernorm(x)

        inp_start_ff_block = x
        x = self.ff(x)

        x = x + inp_start_ff_block
        x = self.enc_blk_end_layernorm(x)
        return [x, mask]

In [23]:
class Encoder(nn.Module):
    def __init__(self, num_layers, num_heads, vocab_size, embed_dim, k_dim, ctx_len, do, device):
        super().__init__()
        self.emb = Embed(vocab_size, embed_dim, ctx_len, do, device)
        self.layer_list = [Encoder_Block(num_heads, embed_dim, k_dim, do, device) for i in range(num_layers)]
        self.layers = nn.Sequential(*self.layer_list)

    def forward(self, input_ids, attention_mask):
        x = self.emb(input_ids)
        mask = att_mask(attention_mask, lookahead=False, cross_att=False, x=None)
        x = self.layers([x, mask])
        return x[0]

In [24]:
enc = Encoder(3, 6, 100, 768, 128, 100, 0.1, device)
enc.to(device)

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(100, 768)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-5): 6 x Attention(
            (query): Linear(in_features=768, out_features=128, bias=True)
            (key): Linear(in_features=768, out_features=128, bias=True)
            (value): Linear(in_features=768, out_features=128, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
  

In [25]:
summary(enc)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         76,800
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    1,536
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              2,362,368
│    │    └─LayerNorm: 3-5                    1,536
│    │    └─Sequential: 3-6                   4,722,432
│    │    └─LayerNorm: 3-7                    1,536
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              2,362,368
│    │    └─LayerNorm: 3-9                    1,536
│    │    └─Sequential: 3-10                  4,722,432
│    │    └─LayerNor

In [26]:
inp = torch.randint(1,100, (4,6))
am = torch.ones(inp.shape)
inp.shape, am.shape

(torch.Size([4, 6]), torch.Size([4, 6]))

In [27]:
out = enc(inp.to(device), am.to(device))
out.shape

torch.Size([4, 6, 768])

In [28]:
class MLM(nn.Module):
    def __init__(self, enc, embed_dim, vocab_size):
        super().__init__()

        self.encoder = enc
        self.mlm_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, input_ids, attention_mask):
        x = self.encoder(input_ids, attention_mask)
        x = self.mlm_layer(x)
        return x

In [29]:
model_mlm = MLM(enc, 768, 36000)
model_mlm.to(device)

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(100, 768)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-5): 6 x Attention(
              (query): Linear(in_features=768, out_features=128, bias=True)
              (key): Linear(in_features=768, out_features=128, bias=True)
              (value): Linear(in_features=768, out_features=128, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=768, out_features=768, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNor

In [30]:
summary(model_mlm)

Layer (type:depth-idx)                             Param #
MLM                                                --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         76,800
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        1,536
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     7,087,872
│    │    └─Encoder_Block: 3-5                     7,087,872
│    │    └─Encoder_Block: 3-6                     7,087,872
├─Linear: 1-2                                      27,684,000
Total params: 49,025,952
Trainable params: 49,025,952
Non-trainable params: 0

In [31]:
out = model_mlm(inp.to(device), am.to(device))
out, out.shape

(tensor([[[ 0.8798, -0.3187, -0.1912,  ...,  0.2267, -0.3346,  0.3053],
          [-0.2505, -0.2263, -0.7480,  ...,  0.2541, -0.5820,  0.7315],
          [ 0.5601, -0.4758,  0.4796,  ..., -1.0208,  0.6172, -0.5813],
          [-0.3299, -0.0785,  0.7356,  ..., -0.2092, -0.1876,  0.2377],
          [-0.0550, -0.1602,  0.9643,  ..., -0.9650,  0.4905, -0.6751],
          [ 0.0243,  0.4226, -0.1921,  ...,  0.0827,  0.5370,  0.2664]],
 
         [[ 0.3910, -0.2775, -0.4450,  ..., -0.4903, -0.1943, -0.5139],
          [ 0.8744,  0.3310, -0.4308,  ..., -1.2726,  0.4207, -0.1840],
          [ 0.3096,  0.5121, -0.0451,  ..., -0.3066, -0.9303,  0.7152],
          [-0.6294, -0.5274,  0.0231,  ..., -0.4083,  0.4094, -1.0037],
          [-0.7904, -0.0071,  0.8717,  ...,  0.4376, -0.9116,  0.5157],
          [ 0.1486, -0.7271,  0.1699,  ..., -0.5983,  0.0058,  0.2841]],
 
         [[-0.0304, -0.3606, -0.3690,  ..., -0.0623, -0.8899,  0.2064],
          [ 0.4639, -0.7311, -1.0043,  ..., -0.5891,  0.17

In [32]:
from datasets import load_dataset

In [35]:
ds_name = 'fancyzhx/ag_news'
ds = load_dataset(ds_name)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [36]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [37]:
df_train = ds['train'].to_pandas()

In [38]:
df_test = ds['test'].to_pandas()

In [39]:
df_train.to_csv('./agnews_train', index=False)
df_test.to_csv('./agnews_test', index=False)

In [40]:
%pwd

'/home/ec2-user/SageMaker/EncoderTasks'

In [41]:
!ls -ltrh ./agnews*

-rw-rw-r-- 1 ec2-user ec2-user  28M Mar 23 05:33 ./agnews_train
-rw-rw-r-- 1 ec2-user ec2-user 1.8M Mar 23 05:33 ./agnews_test


In [42]:
tok_ckpt = 'bert-base-uncased'
orig_tokenizer = AutoTokenizer.from_pretrained(tok_ckpt)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [43]:
orig_tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [44]:
df_train.columns, df_test.columns

(Index(['text', 'label'], dtype='object'),
 Index(['text', 'label'], dtype='object'))

In [45]:
df = pd.concat([df_train, df_test], axis=0)
df

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2
...,...,...
7595,Around the world Ukrainian presidential candid...,0
7596,Void is filled with Clement With the supply of...,1
7597,Martinez leaves bitter Like Roger Clemens did ...,1
7598,5 of arthritis patients in Singapore take Bext...,2


In [46]:
text_list = df['text'].tolist()

In [47]:
len(text_list)

127600

In [48]:
text_list[:2]

["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.']

In [49]:
text_list[3]

'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.'

In [50]:
unq_words = sorted(list(set([w.lower() for t in text_list for w in t.split()])))

In [51]:
len(unq_words)

163946

In [52]:
text_gen = (text_list[i:i+1000] for i in range(0, len(text_list), 1000))

In [53]:
text_gen

<generator object <genexpr> at 0x7fe696be15b0>

In [54]:
news_tokenizer = orig_tokenizer.train_new_from_iterator(text_gen, 9000) #vocab size of 9000

In [55]:
len(orig_tokenizer.tokenize(text_list[0]))

39

In [56]:
len(news_tokenizer.tokenize(text_list[0]))

42

In [57]:
news_tokenizer

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [59]:
news_tokenizer.save_pretrained("./agnews_tokenizer")

('./agnews_tokenizer/tokenizer_config.json',
 './agnews_tokenizer/special_tokens_map.json',
 './agnews_tokenizer/vocab.txt',
 './agnews_tokenizer/added_tokens.json',
 './agnews_tokenizer/tokenizer.json')

In [56]:
#we placed a limit on vocab size to make model size small so that we can train easy peasy

In [57]:
df['text_len'] = df['text'].apply(lambda x: len(x))

In [58]:
df.describe()

,label,text_len
count,127600.000000,127600.000000
mean,1.500000,236.407343
std,1.118038,66.438756
min,0.000000,100.000000
25%,0.750000,196.000000
50%,1.500000,232.000000
75%,2.250000,266.000000
max,3.000000,1012.000000


In [59]:
class NEWS_DS(Dataset):
    def __init__(self, df):
        self.data = df['text']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data = self.data.iloc[idx]
        return data

In [60]:
news_ds = NEWS_DS(df)

In [61]:
len(news_ds)

127600

In [62]:
news_ds[2]

"Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums."

In [63]:
news_tokenizer.encode(news_tokenizer.mask_token)

[2, 4, 3]

In [64]:
orig_tokenizer.encode(news_tokenizer.mask_token)

[101, 103, 102]

In [65]:
def bernoulli_true_false(p):
    # Create a Bernoulli distribution with probability p
    bernoulli_dist = torch.distributions.Bernoulli(torch.tensor([p]))
    # Sample from this distribution and convert 1 to True and 0 to False
    return bernoulli_dist.sample().item() == 1

In [66]:
torch.randint(0, 9000, size=(1,)).item() #.squeeze().item()

3843

In [67]:
def Masking(token, vocab_size):
    # Decide whether to mask this token (15% chance)
    mask = bernoulli_true_false(0.15)

    # If mask is False, immediately return with '[PAD]' label
    if not mask:
        status = "nomask"
        return token, '[PAD]', status

    # If mask is True, proceed with further operations
    # Randomly decide on an operation (10% chance each)
    replace_random = bernoulli_true_false(0.1)
    leave_as_is = bernoulli_true_false(0.1)
    if replace_random:
        _token = torch.randint(0, vocab_size, size=(1,)).item()
        label = token
        status = "random"
    elif leave_as_is:
        _token = token
        label = token
        status = "asis"
    else:
        _token = '[MASK]'
        label = token
        status = "masked"
        
    return _token, label, status

In [68]:
vocab_size = news_tokenizer.vocab_size
vocab_size

9000

In [69]:
text_list[3]

'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.'

In [70]:
news_tokenizer(text_list[3], max_length=256, truncation=True)

{'input_ids': [2, 366, 4363, 70, 426, 4198, 241, 1459, 1823, 5900, 11, 231, 12, 231, 15, 2474, 310, 4363, 104, 426, 5669, 32, 7206, 70, 241, 102, 1459, 5900, 108, 1823, 366, 264, 32, 3596, 2189, 34, 2280, 5430, 592, 1746, 32, 6156, 14, 117, 426, 560, 221, 131, 696, 16, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [71]:
inp = news_tokenizer(text_list[:8]).input_ids
inp

[[2,
  1405,
  134,
  16,
  3934,
  267,
  1104,
  523,
  469,
  102,
  1741,
  11,
  231,
  12,
  231,
  15,
  1551,
  15,
  1066,
  148,
  14,
  1405,
  1456,
  10,
  52,
  6859,
  1187,
  900,
  32,
  4083,
  115,
  5665,
  161,
  15,
  1997,
  6516,
  70,
  14,
  318,
  7661,
  1900,
  381,
  16,
  3],
 [2,
  3668,
  3199,
  2858,
  2057,
  2990,
  8081,
  11,
  231,
  12,
  231,
  15,
  2587,
  2364,
  1286,
  3668,
  3199,
  563,
  14,
  32,
  637,
  230,
  34,
  8389,
  135,
  1739,
  1570,
  15,
  552,
  74,
  132,
  3492,
  5278,
  702,
  32,
  3692,
  4694,
  108,
  102,
  1785,
  1076,
  14,
  230,
  7555,
  4970,
  32,
  209,
  1291,
  70,
  131,
  1009,
  581,
  115,
  102,
  667,
  16,
  3],
 [2,
  426,
  132,
  1414,
  7159,
  605,
  10,
  2279,
  11,
  231,
  12,
  231,
  15,
  5225,
  1647,
  607,
  4322,
  3287,
  32,
  488,
  102,
  1414,
  132,
  102,
  2279,
  135,
  1197,
  318,
  947,
  111,
  32,
  5097,
  275,
  102,
  1089,
  667,
  598,
  458,
  1050,
  102,


In [72]:
inp_mask  = [Masking(i, vocab_size) for il in inp for i in il]
len(inp_mask)

429

In [73]:
no_mask = 0
masked = 0
random = 0
asis = 0
for i, t, s in inp_mask:
    if s == "nomask":
        no_mask+=1
    elif s == 'masked':
        masked+=1
    elif s == "random":
        random+=1
    else:
        asis+=1
        

In [74]:
no_mask, masked, random, asis

(366, 53, 3, 7)

In [75]:
429*0.15

64.35

In [76]:
429*0.15*0.8

51.48

In [77]:
429*0.15*0.1

6.435

In [78]:
#so masking is woring good

In [79]:
class NEWS_DS(Dataset):
    def __init__(self, df):
        self.data = df['text']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]
        return text

In [80]:
news_tokenizer.vocab['[MASK]'], news_tokenizer.vocab['[PAD]']

(4, 0)

In [81]:
def Masking(token, vocab_size):
    # Decide whether to mask this token (15% chance)
    mask = bernoulli_true_false(0.15)

    # If mask is False, immediately return with '[PAD]' label
    if not mask:
        return token, -100

    # If mask is True, proceed with further operations
    # Randomly decide on an operation (10% chance each)
    replace_random = bernoulli_true_false(0.1)
    leave_as_is = bernoulli_true_false(0.1)
    label = token
    if replace_random:
        _token = torch.randint(0, vocab_size, size=(1,)).item()
    elif leave_as_is:
        _token = token
    else:
        _token = 4

    return _token, label

In [82]:
news_tokenizer.vocab[news_tokenizer.pad_token], news_tokenizer.vocab[news_tokenizer.mask_token]

(0, 4)

In [83]:
pad_tokenid = [news_tokenizer.vocab[news_tokenizer.pad_token]]
type(pad_tokenid[0])

int

In [84]:
def collate_fn(batch, tokenizer):
    vocab_size = tokenizer.vocab_size
    pad_tokenid = [tokenizer.vocab[news_tokenizer.pad_token]]
    ignore_loss_id = [-100]  #ignore loss
    ignore_token_id = [0]   #attention mask filler
    x_input_ids, x_am = [], []
    lab_list = []

    for x in batch:
        tokenized = tokenizer(x, max_length=256, truncation=True)
        inp_ids = []
        label_list = []
        for x in tokenized.input_ids:
            tok, lbl = Masking(x, vocab_size)
            inp_ids.append(tok)
            label_list.append(lbl)
        am = tokenized.attention_mask
        x_input_ids.append(inp_ids)
        x_am.append(am)
        lab_list.append(label_list)

    len_enc = [len(l) for l in x_input_ids]
    max_len = max(len_enc)

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    input_ids = pad_tokens_stack(x_input_ids, max_len, pad_tokenid)
    attention_mask = pad_tokens_stack(x_am, max_len, ignore_token_id)
    labels =  pad_tokens_stack(lab_list, max_len, ignore_loss_id)

    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [85]:
train_ds = NEWS_DS(df_train)
test_ds = NEWS_DS(df_test)

In [86]:
len(train_ds), len(test_ds)

(120000, 7600)

In [87]:
torch.cuda.empty_cache()

In [88]:
from functools import partial
import os
wrapper_collate_fn = partial(
    collate_fn,
    tokenizer=news_tokenizer,
    )

train_dl = DataLoader(train_ds, shuffle=False, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)
test_dl = DataLoader(test_ds, shuffle=False, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)

In [89]:
train_iter = iter(train_dl)

In [90]:
i, l = next(train_iter)

In [91]:
unravel = [t for i in l.tolist() for t in i]

In [92]:
len(unravel)

16256

In [93]:
valid_items = [i for i in unravel if i > -100]
valid_items

[16,
 267,
 1066,
 1405,
 32,
 1997,
 7661,
 1900,
 381,
 2858,
 3668,
 563,
 8389,
 16,
 2,
 132,
 231,
 5225,
 1647,
 16,
 2,
 231,
 32,
 70,
 16,
 607,
 111,
 15,
 1375,
 5702,
 1269,
 1509,
 1874,
 2344,
 742,
 271,
 2409,
 3,
 1313,
 217,
 1330,
 712,
 160,
 727,
 108,
 186,
 135,
 221,
 6672,
 160,
 34,
 16,
 37,
 16,
 51,
 102,
 1356,
 6529,
 1286,
 14,
 34,
 6016,
 265,
 2310,
 357,
 3346,
 2310,
 102,
 2165,
 241,
 16,
 3,
 11,
 14,
 4083,
 6516,
 70,
 1900,
 426,
 275,
 193,
 135,
 3474,
 1233,
 12,
 2246,
 16,
 660,
 8229,
 221,
 696,
 607,
 3266,
 3776,
 69,
 4265,
 14,
 1384,
 769,
 102,
 981,
 283,
 2186,
 240,
 462,
 264,
 102,
 5942,
 5197,
 496,
 11,
 102,
 6862,
 131,
 34,
 54,
 16,
 108,
 6678,
 5534,
 111,
 1014,
 2624,
 1312,
 7996,
 325,
 1587,
 168,
 15,
 15,
 34,
 581,
 115,
 523,
 410,
 1038,
 667,
 1263,
 357,
 4583,
 3131,
 484,
 10,
 504,
 1642,
 175,
 3,
 102,
 219,
 646,
 8,
 3081,
 162,
 1843,
 14,
 1510,
 34,
 892,
 4045,
 102,
 723,
 10,
 209,
 3292,
 1

In [94]:
len(valid_items)

1116

In [95]:
16256*.15

2438.4

In [96]:
# so this is all good. next we need to train hte model

In [97]:
from torch.optim import AdamW

In [114]:
def train_model(num_epochs, lrate, model, dl, device, vocab_size):
    epochs = num_epochs
    lr = lrate
    fn_loss = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr)
    grad_accum_steps = 1 #args.grad_accum_steps # 4 - previous static value
    model = model.to(device)

    for i in range(epochs):
        loss_epoch = 0
        n_step = 0
        grad_accum_counter = 1

        for inputs in dl:
            data = inputs[0]
            label = inputs[1]
            batch_size = label.shape[0]
            ctx_size = label.shape[1]
            data = {i: k.to(device) for i, k in data.items()}
            label = label.to(device)
            out = model(**data)
            out = out.view(batch_size * ctx_size, vocab_size)
            label = label.view(batch_size * ctx_size)
            loss = fn_loss(out, label)
            loss = loss / grad_accum_steps
            loss.backward()
            if grad_accum_counter == grad_accum_steps:
                #logger.info(f'Steps: {grad_accum_counter}, adjusting learnable params now')
                optimizer.step()
                optimizer.zero_grad()
                grad_accum_counter = 0
            loss_epoch = loss_epoch + (loss.item() * batch_size * grad_accum_steps)
            if n_step % 100 == 0:
                print(f'Step: {n_step}, Loss: {loss.item()}')
            grad_accum_counter += 1
            n_step += 1
            ##update so that if remaining dataloader runs are less than grad accum steps then at the last run i should
            #gradient update
            #print(f'Completed minibatch loop')
        average_loss = loss_epoch/len(dl.dataset)
        perplexity = math.exp(average_loss) 
        print(f'Epoch: {i} -- Average loss: {average_loss}')
        print(f'Epoch: {i} -- Perplexity: {perplexity}')

    return model, optimizer, average_loss, perplexity


In [99]:
vocab_size = news_tokenizer.vocab_size
vocab_size

9000

In [100]:
BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size, 
                     embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)

In [101]:
BroEncoder

Encoder(
  (emb): Embed(
    (tok_layer): Embedding(9000, 256)
    (pos_layer): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (norm_do): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): Dropout(p=0.1, inplace=False)
    )
  )
  (layers): Sequential(
    (0): Encoder_Block(
      (MHA): Attention_Block(
        (heads): ModuleList(
          (0-7): 8 x Attention(
            (query): Linear(in_features=256, out_features=32, bias=True)
            (key): Linear(in_features=256, out_features=32, bias=True)
            (value): Linear(in_features=256, out_features=32, bias=True)
            (att_do): Dropout(p=0.1, inplace=False)
          )
        )
        (lin_do): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Dropout(p=0.1, inplace=False)
        )
      )
      (mha_blk_end_layernorm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
    

In [102]:
summary(BroEncoder)

Layer (type:depth-idx)                        Param #
Encoder                                       --
├─Embed: 1-1                                  --
│    └─Embedding: 2-1                         2,304,000
│    └─PositionalEncoding: 2-2                --
│    │    └─Dropout: 3-1                      --
│    └─Sequential: 2-3                        --
│    │    └─LayerNorm: 3-2                    512
│    │    └─Dropout: 3-3                      --
├─Sequential: 1-2                             --
│    └─Encoder_Block: 2-4                     --
│    │    └─Attention_Block: 3-4              263,168
│    │    └─LayerNorm: 3-5                    512
│    │    └─Sequential: 3-6                   525,568
│    │    └─LayerNorm: 3-7                    512
│    └─Encoder_Block: 2-5                     --
│    │    └─Attention_Block: 3-8              263,168
│    │    └─LayerNorm: 3-9                    512
│    │    └─Sequential: 3-10                  525,568
│    │    └─LayerNorm: 3-11      

In [103]:
Bro_MLM = MLM(BroEncoder, 256, vocab_size)
Bro_MLM

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNorm(

In [104]:
summary(Bro_MLM)

Layer (type:depth-idx)                             Param #
MLM                                                --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         2,304,000
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        512
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     789,760
│    │    └─Encoder_Block: 3-5                     789,760
│    │    └─Encoder_Block: 3-6                     789,760
│    │    └─Encoder_Block: 3-7                     789,760
│    │    └─Encoder_Block: 3-8                     789,760
│    │    └─Encoder_Block: 3-9                     789,760
├─Linear: 1-2                                      2,313,000
Total params: 9,356,072
Trainable params: 9,356,072
Non-trainable params: 0

In [ ]:
%%time
model, optimizer, average_loss, perplexity = train_model(10, 0.0004, Bro_MLM, train_dl, device, vocab_size)

Step: 0, Loss: 9.216163635253906


In [106]:
model

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNorm(

In [107]:
torch.save({
        'epoch': 10,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': average_loss,
        'device': device
        }, './mlmckpt')


In [108]:
average_loss

3.4532971148173015

In [109]:
%pwd

'/home/ec2-user/SageMaker/EncoderTasks'

In [110]:
!ls -ltrh

total 138M
-rw-rw-r-- 1 ec2-user ec2-user  28M Mar 22 10:51 agnews_train
-rw-rw-r-- 1 ec2-user ec2-user 1.8M Mar 22 10:51 agnews_test
-rw-rw-r-- 1 ec2-user ec2-user 108M Mar 22 13:45 mlmckpt
-rw-rw-r-- 1 ec2-user ec2-user 133K Mar 22 13:48 Encoder_MLM.ipynb


In [112]:
device, perplexity

('cuda', 31.60442412716105)

In [115]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in train_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(train_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Epoch: {i} -- Average loss: {average_loss}')
print(f'Epoch: {i} -- Perplexity: {perplexity}')

Epoch: {'input_ids': tensor([[   2, 1405,  134,  ...,    0,    0,    0],
        [   2, 3668, 3199,  ...,    0,    0,    0],
        [   4,  426,    4,  ...,    0,    0,    0],
        ...,
        [   2, 1759, 4627,  ...,    0,    0,    0],
        [   4,  102,  598,  ...,    0,    0,    0],
        [   4, 1136,   14,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])} -- Average loss: 3.151042181523641
Epoch: {'input_ids': tensor([[   2, 1405,  134,  ...,    0,    0,    0],
        [   2, 3668, 3199,  ...,    0,    0,    0],
        [   4,  426,    4,  ...,    0,    0,    0],
        ...,
        [   2, 1759, 4627,  ...,    0,    0,    0],
        [   4,  102,  598,  ...,    0,    0,    0],
        [   4, 1136,   14,  ...,    0,    0,    0]]), 'attention_mask': tensor([[1, 1,

In [116]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in test_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(train_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Test Average loss: {average_loss}')
print(f'Test Perplexity: {perplexity}')

Test Average loss: 0.20677252238591512
Test Perplexity: 1.2297028101009242


In [146]:
train_ds[12]

'Non-OPEC Nations Should Up Output-Purnomo  JAKARTA (Reuters) - Non-OPEC oil exporters should consider  increasing output to cool record crude prices, OPEC President  Purnomo Yusgiantoro said on Sunday.'

In [120]:
text = "There is so much of the [MASK] pollution nowadays"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)

In [125]:
input_ids

tensor([[   2, 1038,  174,  492, 1420,  115,  102,    4, 7094,  902, 3273, 2051,
            3]], device='cuda:0')

In [121]:
out.shape

torch.Size([1, 13, 9000])

In [122]:
out_max = torch.argmax(out, dim=-1)
out_max

tensor([[   2, 1038,  174,  492, 1420,  115,  102,  186,  115,  902, 1427, 2051,
            3]], device='cuda:0')

In [129]:
news_tokenizer.decode([out_max[0][7].item()])

'new'

In [130]:
text = "I saw a [MASK] building today"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)

In [131]:
input_ids

tensor([[   2,   42, 3740,   34,    4, 2900,  625,    3]], device='cuda:0')

In [133]:
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][4].item()])

'whole'

In [143]:
text = "Oil prices [MASK] to all-time record"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'fall'

In [144]:
text = "Calif. Aims to [MASK] Farm-Related Smog"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'buy'

In [147]:
text = "Non-OPEC oil exporters should consider  [MASK] output to cool record crude prices"
tok = news_tokenizer(text, return_tensors='pt')
input_ids = tok.input_ids.to(device)
mask_index = input_ids[0].tolist().index(4)
attention_mask = tok.attention_mask.to(device)
input_ids.shape, attention_mask.shape
with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask)
out_max = torch.argmax(out, dim=-1)
news_tokenizer.decode([out_max[0][mask_index].item()])

'oil'

##### MLM on imdb ds

In [33]:
ds_name_2 = 'stanfordnlp/imdb'
ds_imdb = load_dataset(ds_name_2)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

unsupervised-00000-of-00001.parquet:   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [34]:
ds_imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [35]:
tokenizer = AutoTokenizer.from_pretrained('./agnews_tokenizer')
tokenizer

BertTokenizerFast(name_or_path='./agnews_tokenizer', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [36]:
df_train = ds_imdb['train'].to_pandas()
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [37]:
df_test = ds_imdb['test'].shuffle(seed=42).select(range(5000)).to_pandas()

In [38]:
df_test['label'].value_counts()

label
1    2506
0    2494
Name: count, dtype: int64

In [39]:
df = pd.concat([df_train, df_test], axis=0)
df

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
4995,This was a pleasant musical about the creation...,1
4996,"OK, I saw this film through Mystery Science Th...",0
4997,It's painfully clear that all effort in this f...,0
4998,The Vietnam War era is certainly far before my...,1


In [40]:
imdb_list = df['text'].tolist()
len(imdb_list)

30000

In [41]:
imdb_list[:4]

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

In [42]:
len_items = [len(tokenizer(i).input_ids) for i in imdb_list]

Token indices sequence length is longer than the specified maximum sequence length for this model (559 > 512). Running this sequence through the model will result in indexing errors


In [68]:
max(len_items)

3660

In [69]:
len_items[:3]

[412, 362, 147]

In [71]:
df['tok_len'] = df['text'].apply(lambda x: len(tokenizer(x).input_ids))

In [72]:
df.describe()

,label,tok_len
count,26000.000000,26000.000000
mean,0.499538,360.043269
std,0.500009,271.187799
min,0.000000,14.000000
25%,0.000000,190.000000
50%,0.000000,266.000000
75%,1.000000,438.250000
max,1.000000,3660.000000


In [73]:
class IMDB_DS(Dataset):
    def __init__(self, df):
        self.data = df['text']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]
        return text

In [75]:
imdb_ds = IMDB_DS(df)

In [77]:
imdb_ds[2]

"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />"

In [78]:
len(imdb_ds)

26000

In [79]:
tokenizer.encode(tokenizer.mask_token)

[2, 4, 3]

In [81]:
tokenizer.vocab['[MASK]'], tokenizer.vocab['[PAD]']

(4, 0)

In [80]:
def bernoulli_true_false(p):
    # Create a Bernoulli distribution with probability p
    bernoulli_dist = torch.distributions.Bernoulli(torch.tensor([p]))
    # Sample from this distribution and convert 1 to True and 0 to False
    return bernoulli_dist.sample().item() == 1

In [82]:
def Masking(token, vocab_size):
    # Decide whether to mask this token (15% chance)
    mask = bernoulli_true_false(0.15)

    # If mask is False, immediately return with '[PAD]' label
    if not mask:
        return token, -100

    # If mask is True, proceed with further operations
    # Randomly decide on an operation (10% chance each)
    replace_random = bernoulli_true_false(0.1)
    leave_as_is = bernoulli_true_false(0.1)
    label = token
    if replace_random:
        _token = torch.randint(0, vocab_size, size=(1,)).item()
    elif leave_as_is:
        _token = token
    else:
        _token = 4

    return _token, label

In [83]:
def collate_fn(batch, tokenizer):
    vocab_size = tokenizer.vocab_size
    pad_tokenid = [tokenizer.vocab[tokenizer.pad_token]]
    ignore_loss_id = [-100]  #ignore loss
    ignore_token_id = [0]   #attention mask filler
    x_input_ids, x_am = [], []
    lab_list = []

    for x in batch:
        tokenized = tokenizer(x, max_length=256, truncation=True)
        inp_ids = []
        label_list = []
        for x in tokenized.input_ids:
            tok, lbl = Masking(x, vocab_size)
            inp_ids.append(tok)
            label_list.append(lbl)
        am = tokenized.attention_mask
        x_input_ids.append(inp_ids)
        x_am.append(am)
        lab_list.append(label_list)

    len_enc = [len(l) for l in x_input_ids]
    max_len = max(len_enc)

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    input_ids = pad_tokens_stack(x_input_ids, max_len, pad_tokenid)
    attention_mask = pad_tokens_stack(x_am, max_len, ignore_token_id)
    labels =  pad_tokens_stack(lab_list, max_len, ignore_loss_id)

    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [84]:
train_ds = IMDB_DS(df_train)
test_ds = IMDB_DS(df_test)

In [85]:
len(train_ds), len(test_ds)

(25000, 1000)

In [86]:
torch.cuda.empty_cache()

In [88]:
tokenizer

BertTokenizerFast(name_or_path='./agnews_tokenizer', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [92]:
from functools import partial
import os
wrapper_collate_fn = partial(
    collate_fn,
    tokenizer=tokenizer,
    )

train_dl = DataLoader(train_ds, shuffle=True, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)
test_dl = DataLoader(test_ds, shuffle=False, batch_size=128,
                      num_workers=os.cpu_count(), collate_fn=wrapper_collate_fn)

In [93]:
train_iter = iter(train_dl)

In [94]:
i, l = next(train_iter)

In [95]:
unravel = [t for i in l.tolist() for t in i]

In [96]:
len(unravel)

32768

In [97]:
valid_items = [i for i in unravel if i > -100]
valid_items

[14,
 115,
 3535,
 115,
 4984,
 16,
 3065,
 3124,
 2546,
 109,
 3748,
 333,
 7833,
 306,
 29,
 1510,
 42,
 7354,
 238,
 787,
 6846,
 241,
 174,
 1091,
 112,
 313,
 3017,
 310,
 2423,
 46,
 175,
 387,
 168,
 375,
 16,
 1,
 4809,
 16,
 102,
 14,
 67,
 12,
 434,
 1056,
 321,
 14,
 102,
 207,
 1232,
 29,
 1951,
 7626,
 102,
 16,
 3994,
 6146,
 102,
 3,
 1224,
 109,
 1665,
 16,
 102,
 245,
 4435,
 552,
 951,
 52,
 6161,
 824,
 1587,
 53,
 111,
 42,
 16,
 102,
 2624,
 99,
 207,
 895,
 488,
 7128,
 1432,
 102,
 2274,
 115,
 52,
 65,
 34,
 182,
 7479,
 182,
 102,
 102,
 351,
 693,
 310,
 159,
 4930,
 158,
 7932,
 102,
 2904,
 658,
 241,
 8007,
 1697,
 2790,
 230,
 1338,
 504,
 64,
 672,
 2948,
 5491,
 2314,
 8,
 23,
 108,
 1471,
 403,
 4771,
 252,
 7113,
 111,
 132,
 4918,
 68,
 421,
 115,
 6896,
 2339,
 1082,
 109,
 3314,
 1241,
 378,
 102,
 109,
 152,
 271,
 238,
 34,
 271,
 102,
 7031,
 16,
 182,
 5342,
 3035,
 132,
 313,
 1224,
 1923,
 1843,
 1852,
 14,
 733,
 3869,
 3597,
 14,
 115,
 3792

In [98]:
len(valid_items)

4154

In [99]:
len(unravel)*0.15

4915.2

In [100]:
from torch.optim import AdamW

In [102]:
model_dict = torch.load('./mlmckpt')
model_dict.keys()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss', 'device'])

In [103]:
for keys in ['epoch', 'loss', 'device']:
    print(model_dict[keys])

10
3.4532971148173015
cuda


In [104]:
vocab_size = tokenizer.vocab_size
vocab_size

9000

In [105]:
BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size, 
                     embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)

In [106]:
Bro_MLM = MLM(BroEncoder, 256, vocab_size)
Bro_MLM

MLM(
  (encoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_end_layernorm): LayerNorm(

In [107]:
summary(Bro_MLM)

Layer (type:depth-idx)                             Param #
MLM                                                --
├─Encoder: 1-1                                     --
│    └─Embed: 2-1                                  --
│    │    └─Embedding: 3-1                         2,304,000
│    │    └─PositionalEncoding: 3-2                --
│    │    └─Sequential: 3-3                        512
│    └─Sequential: 2-2                             --
│    │    └─Encoder_Block: 3-4                     789,760
│    │    └─Encoder_Block: 3-5                     789,760
│    │    └─Encoder_Block: 3-6                     789,760
│    │    └─Encoder_Block: 3-7                     789,760
│    │    └─Encoder_Block: 3-8                     789,760
│    │    └─Encoder_Block: 3-9                     789,760
├─Linear: 1-2                                      2,313,000
Total params: 9,356,072
Trainable params: 9,356,072
Non-trainable params: 0

In [108]:
model_dict.keys()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss', 'device'])

In [110]:
Bro_MLM.load_state_dict(model_dict['model_state_dict'])

<All keys matched successfully>

In [111]:
model = Bro_MLM

In [112]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in train_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(train_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Training Average loss: {average_loss}')
print(f'Training Perplexity: {perplexity}')

Training Average loss: 5.883202411651611
Training Perplexity: 358.95693089933525


Training Average loss: 0.23519211776733398
Training Perplexity: 1.2651518035263254


In [116]:
model, optimizer, average_loss, perplexity = train_model(10, 0.0004, Bro_MLM, train_dl, device, vocab_size)

Step: 0, Loss: 5.840501308441162
Step: 100, Loss: 4.1548991203308105
Epoch: 0 -- Average loss: 4.228756598434448
Epoch: 0 -- Perplexity: 68.63184215782235
Step: 0, Loss: 3.7393600940704346
Step: 100, Loss: 3.5682082176208496
Epoch: 1 -- Average loss: 3.629689563140869
Epoch: 1 -- Perplexity: 37.701110985865846
Step: 0, Loss: 3.553816080093384
Step: 100, Loss: 3.3739774227142334
Epoch: 2 -- Average loss: 3.395516984024048
Epoch: 2 -- Perplexity: 29.830071159472038
Step: 0, Loss: 3.2187609672546387
Step: 100, Loss: 3.213336706161499
Epoch: 3 -- Average loss: 3.2408977336120603
Epoch: 3 -- Perplexity: 25.556654519825276
Step: 0, Loss: 3.1452534198760986
Step: 100, Loss: 3.119072437286377
Epoch: 4 -- Average loss: 3.1306783029937746
Epoch: 4 -- Perplexity: 22.889500294535207
Step: 0, Loss: 2.9917099475860596
Step: 100, Loss: 3.0777344703674316
Epoch: 5 -- Average loss: 3.0353518660736083
Epoch: 5 -- Perplexity: 20.808298345729206
Step: 0, Loss: 2.9388041496276855
Step: 100, Loss: 3.0187602

In [120]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in test_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(test_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Test Average loss: {average_loss}')
print(f'Test Perplexity: {perplexity}')

Test Average loss: 2.885202169418335
Test Perplexity: 17.907187515273186


In [122]:
#test loss and perplexity is in line with training dataset
#Let us bake the model for 5 more epochs

In [123]:
model, optimizer, average_loss, perplexity = train_model(5, 0.0004, model, train_dl, device, vocab_size)

Step: 0, Loss: 2.8169198036193848
Step: 100, Loss: 2.7115659713745117
Epoch: 0 -- Average loss: 2.759156516799927
Epoch: 0 -- Perplexity: 15.786521665098821
Step: 0, Loss: 2.763066530227661
Step: 100, Loss: 2.70294189453125
Epoch: 1 -- Average loss: 2.714950934295654
Epoch: 1 -- Perplexity: 15.103868971744802
Step: 0, Loss: 2.7801594734191895
Step: 100, Loss: 2.627650737762451
Epoch: 2 -- Average loss: 2.6722273762512208
Epoch: 2 -- Perplexity: 14.47216828368427
Step: 0, Loss: 2.7393577098846436
Step: 100, Loss: 2.5838873386383057
Epoch: 3 -- Average loss: 2.6409716455078125
Epoch: 3 -- Perplexity: 14.026826091109776
Step: 0, Loss: 2.6095361709594727
Step: 100, Loss: 2.6043498516082764
Epoch: 4 -- Average loss: 2.615363771820068
Epoch: 4 -- Perplexity: 13.67218903047144


In [124]:
model.eval()
fn_loss = nn.CrossEntropyLoss()
model = model.to(device)
loss_epoch = 0
for inputs in test_dl:
    data = inputs[0]
    label = inputs[1]
    batch_size = label.shape[0]
    ctx_size = label.shape[1]
    data = {i: k.to(device) for i, k in data.items()}
    label = label.to(device)
    with torch.no_grad():
        out = model(**data)
    out = out.view(batch_size * ctx_size, vocab_size)
    label = label.view(batch_size * ctx_size)
    loss = fn_loss(out, label)
    loss_epoch = loss_epoch + (loss.item() * batch_size)
average_loss = loss_epoch/len(test_dl.dataset)
perplexity = math.exp(average_loss) 
print(f'Test Average loss: {average_loss}')
print(f'Test Perplexity: {perplexity}')

Test Average loss: 2.760401731491089
Test Perplexity: 15.806191517851754


In [125]:
torch.save({
        'epoch': 15,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': average_loss,
        'device': device
        }, './news_imdb_mlm.pt')


In [43]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [44]:
df_train['label'].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

In [45]:
df_test['label'].value_counts()

label
1    2506
0    2494
Name: count, dtype: int64

In [46]:
import math

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))    #1
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [47]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [48]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):     #1
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:    #2
            replace_linear_with_lora(module, rank, alpha)
#1 Replaces the Linear layer with LinearWithLoRA

In [49]:
#summary(model)

In [50]:
#replace_linear_with_lora(model, rank=16, alpha=16)
#total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
#print(f"Total trainable LoRA parameters: {total_params:,}")

In [51]:
#summary(model)

In [52]:
#model

In [53]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)
#print(model)

In [54]:
#model.mlm_layer

In [132]:
class Bro_Classification_Model(nn.Module):
    def __init__(self, max_length, vocab_size, device):
        super().__init__()
        self.BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size,
                                  embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)
        self.Bro_MLM = MLM(self.BroEncoder, 256, vocab_size)
        model_dict = torch.load('./news_imdb_mlm.pt')
        self.Bro_MLM.load_state_dict(model_dict['model_state_dict'])
        print("Model loaded from checkpoint successfully")

        self.Bro_MLM.mlm_layer = nn.Sequential(
            nn.Linear(max_length, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(128, 2)
        )
        total_params = sum(p.numel() for p in self.Bro_MLM.parameters() if p.requires_grad)
        print(f"Total trainable LoRA and classification parameters: {total_params:,}")


    def forward(self, input_ids, attention_mask):
        x = self.BroEncoder(input_ids, attention_mask)
        x = x.mean(dim=1)
        x = self.Bro_MLM.mlm_layer(x)
        return x

In [103]:
class Bro_Classification_Model_skip(nn.Module):
    def __init__(self, max_length, vocab_size, device):
        super().__init__()
        self.BroEncoder = Encoder(num_layers=6, num_heads=8, vocab_size=vocab_size,
                                  embed_dim=256, k_dim=32, ctx_len=256, do=0.1, device=device)
        self.Bro_MLM = MLM(self.BroEncoder, 256, vocab_size)
        model_dict = torch.load('./news_imdb_mlm.pt')
        self.Bro_MLM.load_state_dict(model_dict['model_state_dict'])
        print("Model loaded from checkpoint successfully")
        #total_params = sum(p.numel() for p in self.Bro_MLM.parameters() if p.requires_grad)
        #print(f"Total trainable parameters before freezing: {total_params:,}")
        #for param in self.Bro_MLM.parameters():
        #    param.requires_grad = False
        #total_params = sum(p.numel() for p in self.Bro_MLM.parameters() if p.requires_grad)
        #print(f"Total trainable parameters after freezing: {total_params:,}")
        #replace_linear_with_lora(self.Bro_MLM, rank=16, alpha=32)
        self.Bro_MLM.mlm_layer = nn.Sequential(
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(64, 1)
        )
        self.gen_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.05),
            )
        self.classifier = nn.Linear(64, 2)
        total_params = sum(p.numel() for p in self.Bro_MLM.parameters() if p.requires_grad)
        print(f"Total trainable LoRA and classification parameters: {total_params:,}")


    def forward(self, input_ids, attention_mask):
        x = self.BroEncoder(input_ids, attention_mask)
        #x = x.mean(dim=1)
        x = self.Bro_MLM.mlm_layer(x)
        #print(x.shape)
        x = self.gen_layer(x)
        #print(x.shape)
        #x = x + y
        x = self.classifier(x)
        return x

In [133]:
vocab_size = tokenizer.vocab_size
vocab_size

9000

In [134]:
bro_model = Bro_Classification_Model(256, vocab_size, device)

Model loaded from checkpoint successfully
Total trainable LoRA and classification parameters: 7,076,482


In [135]:
summary(bro_model)

Layer (type:depth-idx)                                  Param #
Bro_Classification_Model                                --
├─Encoder: 1-1                                          --
│    └─Embed: 2-1                                       --
│    │    └─Embedding: 3-1                              2,304,000
│    │    └─PositionalEncoding: 3-2                     --
│    │    └─Sequential: 3-3                             512
│    └─Sequential: 2-2                                  --
│    │    └─Encoder_Block: 3-4                          789,760
│    │    └─Encoder_Block: 3-5                          789,760
│    │    └─Encoder_Block: 3-6                          789,760
│    │    └─Encoder_Block: 3-7                          789,760
│    │    └─Encoder_Block: 3-8                          789,760
│    │    └─Encoder_Block: 3-9                          789,760
├─MLM: 1-2                                              7,043,072
│    └─Encoder: 2-3                                     (recursiv

In [137]:
bro_model

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [138]:
t = torch.randint(0, 100, (2,256))
t

tensor([[82, 99, 21, 10, 93, 17, 65, 38,  3, 71, 35, 39, 76, 73, 75, 57, 50, 32,
         42, 49, 58, 70, 68, 12, 62, 59, 91, 99, 17,  0, 57, 93, 46, 30, 24, 69,
         84, 16, 90, 14, 92, 94, 97,  5, 45, 68, 93, 26, 42, 67, 54, 92, 37, 44,
         84, 63, 76, 57, 14, 63, 98, 45, 94, 69, 18, 22, 90, 44, 66, 20, 18, 32,
         30,  2, 83, 67, 34, 69, 61, 92, 28, 83,  3,  8, 26, 42, 41, 93, 11, 13,
          2, 60, 68, 58, 25, 75, 79, 71, 40, 88, 37, 77, 91, 42, 10, 81, 79, 74,
         27, 21, 62, 61, 89, 23, 89, 53, 91, 86, 75, 97, 14, 44, 15, 14, 94, 44,
         90,  8, 40, 72, 91, 44, 73, 61, 88,  9, 34, 74, 11, 10, 57, 77, 15, 63,
         43, 47, 54, 28, 72, 57, 12, 45, 37, 19, 56, 72, 10, 16, 92, 19, 51, 82,
         96, 53, 56, 79, 47, 46, 84, 46, 79, 54,  9, 33, 47, 25, 93, 94, 31, 90,
         69, 94, 60, 16, 12, 70, 92, 53,  8, 36, 14, 84, 26, 46, 27,  1, 75, 88,
         15, 93, 86, 38, 56,  7, 87, 61, 33, 50, 53,  2, 15, 96, 85, 92, 53,  7,
         44, 25, 65, 42, 30,

In [139]:
am = torch.randint(1, 2, (2,256))
am

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [140]:
t.shape

torch.Size([2, 256])

In [141]:
t, am= t.to(device), am.to(device)

In [142]:
bro_model.to(device)

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [143]:
out = bro_model(t, am)
out, out.shape

(tensor([[ 0.3597,  0.1926],
         [-0.2961, -0.2147]], device='cuda:0', grad_fn=<AddmmBackward0>),
 torch.Size([2, 2]))

In [144]:
tokenizer = AutoTokenizer.from_pretrained('./agnews_tokenizer/')
tokenizer

BertTokenizerFast(name_or_path='./agnews_tokenizer/', vocab_size=9000, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [145]:
df_train

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [146]:
df_train['label'].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

In [147]:
df_test['label'].value_counts()

label
1    2506
0    2494
Name: count, dtype: int64

In [148]:
class IMDB_DS(Dataset):
    def __init__(self, df, max_len):
        self.data = df['text']
        self.label = df['label']
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = self.data.iloc[idx]
        y = torch.tensor(self.label.iloc[idx])
        tokenized = tokenizer(x, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt')
        input_ids = tokenized.input_ids.squeeze()
        attention_mask = tokenized.attention_mask.squeeze()
        return {'input_ids': input_ids, 'attention_mask': attention_mask}, y

In [149]:
train_ds = IMDB_DS(df_train, 256)
test_ds = IMDB_DS(df_test, 256)

In [150]:
train_ds[0]

({'input_ids': tensor([   2,   42, 7870,  104,   42,  285, 3341,  839,   15, 7698,  241, 1428,
          1471, 2135, 1282,  115,  434,  102, 5938,  182, 5583,  104,  207,  591,
           207,  238,  343, 1392,  108, 7942,   95,   16,   42, 1653, 5820,  182,
           166,  343,  207,  238, 5374,  217,   54,   16,   52,   16, 1406,   68,
           838,  207, 1025, 4288,  111, 1176,  397,  890,   14, 1038, 2676,   65,
          1227,   34, 4203,  115, 8245, 5635,    6, 3692,    6,   42, 4330,  566,
           111, 1929,  397,  135, 7744, 1664,   16,    1,  333,   17,    1,    1,
           333,   17,    1,  102, 3787,  174, 1682,  104, 1545,   34, 2658, 6730,
          8366, 7092, 2648, 6060,   70,  504, 2250,  111, 6823, 5544, 1541,  400,
           488, 1561,   16,  108, 6642, 1541, 2250,  111, 2301,  951, 4958,   68,
           111, 1739,  693, 8031,  115, 6699,  427,  131,  938,  102, 3053, 3724,
            65, 3460,  488, 3564, 1788, 2777, 1897,  162,  102, 5408,  453,  132,
   

In [151]:
test_ds[0]

({'input_ids': tensor([   2,    1,  333,   17,    1,    1,  333,   17,    1,  591,   42, 4633,
           155, 1733,  816,  104,  169, 7870,  104,   34, 2019,  187,  263, 2039,
            14,   42, 3460,   42,  238,  108,  135,  117, 3074,  109, 2731, 3433,
          3894,  132,  115, 3121, 7980,  146, 4989,   65, 6112,   71,  238,  108,
           207,   14,  492,  938,  592,  312, 3759,   31,    1,  333,   17,    1,
             1,  333,   17,    1, 3065, 4602,   14, 6034,   14,   42, 1356, 1553,
           182,  397, 3894,  238,  488,   34, 2019,  187,  845, 3795, 4432, 1901,
           733,  263, 2039,   16,   42, 3312, 5488,  109,  132, 6190,   10,   53,
          1810, 2464,  779,  264,  102, 3339, 2344,   16,  436,   73,  824, 2175,
            65,   14, 6436,   70,  132,  777, 2944, 5844,   14,  135, 4705,  219,
          1897,   34, 6664, 7310, 1091,  576,  132,  224, 5283, 2134, 5943, 3339,
             5,  436,   73,  824, 3421,   14,  135, 1227, 4945,  132, 1564,  161,
   

In [152]:
train_dl = DataLoader(train_ds, shuffle=True, batch_size=32, num_workers=os.cpu_count())
test_dl = DataLoader(test_ds, shuffle=False, batch_size=32, num_workers=os.cpu_count())

In [153]:
len(train_dl.dataset), len(test_dl.dataset)

(25000, 5000)

In [154]:
i, l = next(iter(train_dl))
i.keys, i['input_ids'].shape, i['attention_mask'].shape, l.shape

(<function dict.keys>,
 torch.Size([32, 256]),
 torch.Size([32, 256]),
 torch.Size([32]))

In [155]:
i = {k: v.to(device) for k, v in i.items()}
i

{'input_ids': tensor([[   2,  135,  621,  ...,   52, 1315,    3],
         [   2, 1510, 5988,  ...,  132, 1356,    3],
         [   2, 3077, 1925,  ...,    0,    0,    0],
         ...,
         [   2,   42,  400,  ...,    0,    0,    0],
         [   2, 6477, 6441,  ...,  397,  174,    3],
         [   2,  397,  174,  ...,    0,    0,    0]], device='cuda:0'),
 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [156]:
out = bro_model(**i)
out.shape

torch.Size([32, 2])

In [157]:
torch.cuda.empty_cache()

In [170]:
def test(model, dl, device, vocab_size):
    model.eval()
    fn_loss = nn.CrossEntropyLoss()
    model = model.to(device)

    loss_epoch = 0
    accuracy = 0
    for inputs in dl:
        data = inputs[0]
        label = inputs[1]
        batch_size = data['input_ids'].shape[0]
        ctx_size = data['input_ids'].shape[1]
        data = {i: k.to(device) for i, k in data.items()}
        label = label.to(device)
        with torch.no_grad():
            out = model(**data)
        #out = out.view(batch_size * ctx_size, vocab_size)
        #label = label.view(batch_size * ctx_size)
        loss = fn_loss(out, label)
        loss_epoch = loss_epoch + (loss.item() * batch_size)
        accuracy += (torch.argmax(out, dim=1) == label).sum().item()
    average_loss = loss_epoch/len(dl.dataset)
    average_acc = accuracy/len(dl.dataset)
    print(f'Average Test loss: {average_loss}')
    print(f'Average Test Accuracy: {average_acc}')
    return average_loss, average_acc

In [171]:
def train_model(num_epochs, lrate, model, dl, test_dl, device, vocab_size):
    epochs = num_epochs
    lr = lrate
    fn_loss = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr)
    grad_accum_steps = 1 #args.grad_accum_steps # 4 - previous static value
    model = model.to(device)
    base_test_acc = 0.84

    for i in range(epochs):
        loss_epoch = 0
        n_step = 0
        grad_accum_counter = 1
        accuracy = 0
        model.train()
        

        for inputs in dl:
            data = inputs[0]
            label = inputs[1]
            batch_size = data['input_ids'].shape[0]
            ctx_size = data['input_ids'].shape[1]
            data = {i: k.to(device) for i, k in data.items()}
            label = label.to(device)
            out = model(**data)
            #out = out.view(batch_size * ctx_size, vocab_size)
            #label = label.view(batch_size * ctx_size)
            loss = fn_loss(out, label)
            loss = loss / grad_accum_steps
            loss.backward()
            if grad_accum_counter == grad_accum_steps:
                optimizer.step()
                optimizer.zero_grad()
                grad_accum_counter = 0
            loss_epoch = loss_epoch + (loss.item() * batch_size * grad_accum_steps)
            accuracy += (torch.argmax(out, dim=1) == label).sum().item()
            if n_step % 200 == 0:
                print(f'Step: {n_step}, Loss: {loss.item()}')
            grad_accum_counter += 1
            n_step += 1

        average_loss = loss_epoch/len(dl.dataset)
        average_acc = accuracy/len(dl.dataset)
        print(f'Epoch: {i} -- Average training Loss: {average_loss}')
        print(f'Epoch: {i} -- Average training Accuracy: {average_acc}')
        avg_test_loss, avg_test_acc = test(model, test_dl, device, vocab_size)
        if avg_test_acc > base_test_acc:
            base_test_acc = avg_test_acc
            print("saving model")
            torch.save({
                'epoch': i,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_test_loss,
                'accuracy': avg_test_acc,
                'device': device
                }, './imdb_class_model.pt')

    return model, optimizer, average_loss, average_acc

In [172]:
torch.cuda.empty_cache()

In [173]:
bro_model

Bro_Classification_Model(
  (BroEncoder): Encoder(
    (emb): Embed(
      (tok_layer): Embedding(9000, 256)
      (pos_layer): PositionalEncoding(
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm_do): Sequential(
        (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (1): Dropout(p=0.1, inplace=False)
      )
    )
    (layers): Sequential(
      (0): Encoder_Block(
        (MHA): Attention_Block(
          (heads): ModuleList(
            (0-7): 8 x Attention(
              (query): Linear(in_features=256, out_features=32, bias=True)
              (key): Linear(in_features=256, out_features=32, bias=True)
              (value): Linear(in_features=256, out_features=32, bias=True)
              (att_do): Dropout(p=0.1, inplace=False)
            )
          )
          (lin_do): Sequential(
            (0): Linear(in_features=256, out_features=256, bias=True)
            (1): Dropout(p=0.1, inplace=False)
          )
        )
        (mha_blk_en

In [163]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=10, lrate=0.0003,
                                                          model=bro_model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.8127052783966064
Step: 200, Loss: 0.46270430088043213
Step: 400, Loss: 0.5631912350654602
Step: 600, Loss: 0.6299863457679749
Epoch: 0 -- Average training Loss: 0.4378236967277527
Epoch: 0 -- Average training Accuracy: 0.79732
Average Test loss: 0.3767437571048737
Average Test Accuracy: 0.8342
saving model
Step: 0, Loss: 0.3684602975845337
Step: 200, Loss: 0.7427643537521362
Step: 400, Loss: 0.2368483990430832
Step: 600, Loss: 0.3925231993198395
Epoch: 1 -- Average training Loss: 0.355702325425148
Epoch: 1 -- Average training Accuracy: 0.84464
Average Test loss: 0.36056462383270266
Average Test Accuracy: 0.8436
saving model
Step: 0, Loss: 0.15949581563472748
Step: 200, Loss: 0.2605215311050415
Step: 400, Loss: 0.2404199242591858
Step: 600, Loss: 0.26854878664016724
Epoch: 2 -- Average training Loss: 0.30536746253013614
Epoch: 2 -- Average training Accuracy: 0.86952
Average Test loss: 0.3376775577545166
Average Test Accuracy: 0.8464
saving model
Step: 0, Loss: 0.2564046

In [174]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=6, lrate=0.0003,
                                                          model=model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.16213944554328918
Step: 200, Loss: 0.041101571172475815
Step: 400, Loss: 0.16001741588115692
Step: 600, Loss: 0.1139909103512764
Epoch: 0 -- Average training Loss: 0.09906813566178084
Epoch: 0 -- Average training Accuracy: 0.96152
Average Test loss: 0.5858877855777741
Average Test Accuracy: 0.83
Step: 0, Loss: 0.027473606169223785
Step: 200, Loss: 0.09507177770137787
Step: 400, Loss: 0.01149538904428482
Step: 600, Loss: 0.01848793774843216
Epoch: 1 -- Average training Loss: 0.08543457690298557
Epoch: 1 -- Average training Accuracy: 0.96728
Average Test loss: 0.695128291913867
Average Test Accuracy: 0.8346
Step: 0, Loss: 0.06963293999433517
Step: 200, Loss: 0.08692055195569992
Step: 400, Loss: 0.15298910439014435
Step: 600, Loss: 0.267459899187088
Epoch: 2 -- Average training Loss: 0.08575014843076467
Epoch: 2 -- Average training Accuracy: 0.9678
Average Test loss: 0.6013309459447861
Average Test Accuracy: 0.8286
Step: 0, Loss: 0.05309141427278519
Step: 200, Loss: 0.059

## following experiment gave best results. original bro model using single classification sequential and 0.1 dropout
#tomorrow try out once again using 0.05 dropout, no dropout and then 0.1 save each of the model
#model is learning well and very good rsults on test but kind of overfitting. we tried skip connections. it did not
#help. So perhaps it is more of an issue with model to originally trained on more data than we are using riggt now
#perhaps merge validation dataset with train and then use test data for validation but it will take lot of time 
#to train. Worth trying thought to rule out data hungriness.. see you tomorrow

In [115]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=10, lrate=0.0003,
                                                          model=bro_model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.29630622267723083
Step: 200, Loss: 0.3767191767692566
Step: 400, Loss: 0.27117452025413513
Step: 600, Loss: 0.4704580008983612
Epoch: 0 -- Average training Loss: 0.3578528818130493
Epoch: 0 -- Average training Accuracy: 0.84368
Average Test loss: 0.3778113238811493
Average Test Accuracy: 0.823
Step: 0, Loss: 0.41369813680648804
Step: 200, Loss: 0.24925951659679413
Step: 400, Loss: 0.22235561907291412
Step: 600, Loss: 0.221827894449234
Epoch: 1 -- Average training Loss: 0.3090490364646912
Epoch: 1 -- Average training Accuracy: 0.86836
Average Test loss: 0.34620957720279694
Average Test Accuracy: 0.838
Step: 0, Loss: 0.2785033881664276
Step: 200, Loss: 0.29370978474617004
Step: 400, Loss: 0.38096505403518677
Step: 600, Loss: 0.235026016831398
Epoch: 2 -- Average training Loss: 0.2738749547481537
Epoch: 2 -- Average training Accuracy: 0.88496
Average Test loss: 0.8523226054310798
Average Test Accuracy: 0.769
Step: 0, Loss: 0.2896747291088104
Step: 200, Loss: 0.29275953769

In [116]:
torch.cuda.empty_cache()

In [148]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=10, lrate=0.0003,
                                                          model=bro_model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.7909785509109497
Step: 200, Loss: 0.3639296591281891
Step: 400, Loss: 0.38096943497657776
Step: 600, Loss: 0.555366575717926
Epoch: 0 -- Average training Loss: 0.4390769592189789
Epoch: 0 -- Average training Accuracy: 0.79572
Average Test loss: 0.40017609858512876
Average Test Accuracy: 0.8188
Step: 0, Loss: 0.49086111783981323
Step: 200, Loss: 0.30750951170921326
Step: 400, Loss: 0.3301372230052948
Step: 600, Loss: 0.32998254895210266
Epoch: 1 -- Average training Loss: 0.34987502027511597
Epoch: 1 -- Average training Accuracy: 0.84496
Average Test loss: 0.4015485887050629
Average Test Accuracy: 0.8232
Step: 0, Loss: 0.32598480582237244
Step: 200, Loss: 0.1787509322166443
Step: 400, Loss: 0.5574291944503784
Step: 600, Loss: 0.2652207314968109
Epoch: 2 -- Average training Loss: 0.3022260072517395
Epoch: 2 -- Average training Accuracy: 0.86988
Average Test loss: 0.446350325012207
Average Test Accuracy: 0.8174
Step: 0, Loss: 0.3133648633956909
Step: 200, Loss: 0.281174182

In [191]:
#thrd run

In [193]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=10, lrate=0.0003,
                                                          model=bro_model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.8423508405685425
Step: 200, Loss: 0.6513998508453369
Step: 400, Loss: 0.5072464942932129
Step: 600, Loss: 0.4617939293384552
Epoch: 0 -- Average training Loss: 0.44616307553291323
Epoch: 0 -- Average training Accuracy: 0.79076
Average Test loss: 0.660544674873352
Average Test Accuracy: 0.7104
Step: 0, Loss: 0.3452816307544708
Step: 200, Loss: 0.38347098231315613
Step: 400, Loss: 0.4230605959892273
Step: 600, Loss: 0.40187007188796997
Epoch: 1 -- Average training Loss: 0.351481398563385
Epoch: 1 -- Average training Accuracy: 0.84268
Average Test loss: 0.4341929154396057
Average Test Accuracy: 0.8034
Step: 0, Loss: 0.2827933132648468
Step: 200, Loss: 0.22288374602794647
Step: 400, Loss: 0.414408802986145
Step: 600, Loss: 0.2525902986526489
Epoch: 2 -- Average training Loss: 0.3034148890638351
Epoch: 2 -- Average training Accuracy: 0.86924
Average Test loss: 0.35769237060546877
Average Test Accuracy: 0.8422
Step: 0, Loss: 0.10430754721164703
Step: 200, Loss: 0.12726598978

In [194]:
torch.save({
        'epoch': 10,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': average_loss,
        'device': device
        }, './imdb_skip_bro.pt')


In [195]:
model, optimizer, average_loss, average_acc = train_model(num_epochs=5, lrate=0.001,
                                                          model=model, dl=train_dl,
                                                          test_dl=test_dl, device=device,
                                                          vocab_size=2)

Step: 0, Loss: 0.08310122042894363
Step: 200, Loss: 0.39325278997421265
Step: 400, Loss: 0.3325696885585785
Step: 600, Loss: 0.2620108425617218
Epoch: 0 -- Average training Loss: 0.3049830710029602
Epoch: 0 -- Average training Accuracy: 0.86716
Average Test loss: 1.6374860614776612
Average Test Accuracy: 0.5826
Step: 0, Loss: 0.2888806164264679
Step: 200, Loss: 0.3544050455093384
Step: 400, Loss: 0.3804762363433838
Step: 600, Loss: 0.30437350273132324
Epoch: 1 -- Average training Loss: 0.2651524244403839
Epoch: 1 -- Average training Accuracy: 0.88664
Average Test loss: 0.8935859023094177
Average Test Accuracy: 0.6756
Step: 0, Loss: 0.14580285549163818
Step: 200, Loss: 0.1806003600358963
Step: 400, Loss: 0.20146822929382324
Step: 600, Loss: 0.5246925354003906
Epoch: 2 -- Average training Loss: 0.2551437360107899
Epoch: 2 -- Average training Accuracy: 0.89432
Average Test loss: 0.5087716403961182
Average Test Accuracy: 0.8106
Step: 0, Loss: 0.17919081449508667
Step: 200, Loss: 0.16681128